In [ ]:
import os
import joblib
import pandas as pd

import mlflow
import mlflow.sklearn

#!pip install minio mlflow evidently
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

from mlflow.tracking import MlflowClient


In [ ]:
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)



In [ ]:
iris = load_iris(as_frame=True)

X = iris.data
y = iris.target
print(X.head())
print()
print("Samples:", len(X))
print("Classes:", iris.target_names)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
)
model.fit(X_train, y_train)
    

In [ ]:
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.3f}")
print()
print(classification_report(y_test, predictions))


In [ ]:
reference_data = X_train.copy()
reference_data.to_csv(
    "data/reference_data.csv",
    index=False
)
print("Saved reference data:", "data/reference_data.csv")
print(reference_data.head())

In [ ]:
serving_model_path = "models/iris_model.pkl"
joblib.dump(model, serving_model_path)
print("Saved serving model:", serving_model_path)

In [ ]:
!ls -la models


In [ ]:
MLFLOW_TRACKING_URI = "http://mlflow-server.mlflow.svc.cluster.local:8080"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("openshift-ai-drift-blog-1")



In [ ]:
with mlflow.start_run(run_name="iris-randomforest-baseline") as run:
    mlflow.log_param("algorithm", "RandomForestClassifier")
    mlflow.log_param("n_estimators",100)
    mlflow.log_param("random_state", 42)
    mlflow.log_metric("accuracy", accuracy)
    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="iris-model",
        registered_model_name="iris-model"
    )

    mlflow.log_artifact(
        serving_model_path,
        artifact_path="deployment/sklearn"
    )                        

    mlflow.log_artifact(
        "data/reference_data.csv",
        artifact_path="monitoring/reference-data"
    )
    print("Run ID:", run.info.run_id)
    print("Experiment ID:", run.info.experiment_id)
    